# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` for transparency and reproducibility.

### Dataset Source
The dataset is described via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded.")
print(f"Name      : {getattr(metadata, 'name', None)}")
print(f"Version   : {getattr(metadata, 'version', None)}")
print(f"Published : {getattr(metadata, 'datePublished', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Description:\n{getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets, their `@id`, and field `@id`s (using the Croissant metadata)
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', None)}")
    print(f"  Fields:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    - {field['@id']} (label: {field.get('name', None)})")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use record set and field `@id`s from above.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# List all the recordSet @id values
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids detected:", record_set_ids, "\n")

dataframes = {}
for rs_id in record_set_ids:
    try:
        # Load all records for this record set into a DataFrame
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"{rs_id}: loaded {df.shape[0]} rows, {df.shape[1]} columns.")
            print("Columns:", df.columns.tolist()[:10], "...\n")
        else:
            print(f"{rs_id}: no records found.")
    except Exception as e:
        print(f"Error loading {rs_id}:", e)

# For demonstration, pick the first record set (if available)
if len(record_set_ids) > 0:
    example_rs_id = record_set_ids[0]
    print(f"\nShowing first 5 records from '{example_rs_id}':")
    if example_rs_id in dataframes:
        display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and aggregation, referencing all columns by their field `@id`.

In [ ]:
# For EDA, let us select a DataFrame with numeric data (pick the first suitable one)
import numpy as np
eda_done = False
numeric_field = None
group_field = None
example_df = None
rs_id = None

for rid, df in dataframes.items():
    numeric_fields = [col for col in df.select_dtypes(include=[np.number]).columns]
    if numeric_fields:
        # Select the first record set and numeric field
        example_df = df
        rs_id = rid
        numeric_field = numeric_fields[0]
        # Try to get a group-able column (categorical)
        candidate_group_fields = df.select_dtypes(include=[object]).columns.tolist()
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        break

if example_df is not None:
    print(f"Example DataFrame columns: {example_df.columns.tolist()}")
    print(f"Using numeric field (@id): {numeric_field}")
    if group_field:
        print(f"Using group field (@id): {group_field}")

    # Filtering
    threshold = example_df[numeric_field].quantile(0.75)
    filtered_df = example_df[example_df[numeric_field] > threshold].copy()
    print(f"\nFiltered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nHead of filtered and normalized:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Grouping
    if group_field is not None and group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped.head())
else:
    print("No numeric fields found in dataset for EDA.")

## 5. Visualization
Visualize distributions and relationships among fields identified above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_df is not None and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(example_df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouped data is available
    if group_field and group_field in example_df.columns:
        plt.figure(figsize=(10,4))
        order = example_df[group_field].value_counts().index[:10]
        sns.boxplot(data=example_df, x=group_field, y=numeric_field, order=order)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR^2 Mass Spectrometry dataset using `mlcroissant`, referencing record sets and fields by their `@id` throughout.

- We listed all available record sets and their field `@id`s as documented in the schema.
- We loaded and previewed the data, applying standard EDA steps, and visualized relevant distributions and relationships.

Further domain-specific analysis can now be performed on the `dataframes` loaded, guided by schema exploration and metadata reference.
